###TASK 1 — Multi-Source Enterprise Data Integration

Install dependencies and Import libraries

In [ ]:
!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q langchain
!pip install -q transformers
!pip install -q pytesseract
!pip install -q pillow
!pip install -q python-docx
!pip install -q accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.3 MB/s eta 0:00:00


In [ ]:
import os
import time
import numpy as np
import pdfplumber
import faiss
import pytesseract

from PIL import Image
from google.colab import files

from sentence_transformers import SentenceTransformer

!pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

from transformers import pipeline

In [ ]:
uploaded=files.upload()

Saving POLICY_WORDINGS_b6b468ad34.pdf to POLICY_WORDINGS_b6b468ad34 (2).pdf


PDF extraction

In [ ]:
def extract_pdf(file):

    text=""

    with pdfplumber.open(file) as pdf:

        for page in pdf.pages:

            page_text=page.extract_text()

            if page_text:

                text+=page_text

    return text


DOCX extraction

In [ ]:
from docx import Document

def extract_docx(file):

    doc=Document(file)

    text=[]

    for para in doc.paragraphs:

        text.append(para.text)

    return "\n".join(text)

Image OCR extraction

In [ ]:
def extract_image(file):

    image=Image.open(file)

    text=pytesseract.image_to_string(image)

    return text

Create metadata

In [ ]:
documents=[]

for file in uploaded.keys():

    if file.endswith(".pdf"):

        text=extract_pdf(file)

        filetype="pdf"

    elif file.endswith(".docx"):

        text=extract_docx(file)

        filetype="docx"

    elif file.endswith((".png",".jpg",".jpeg")):

        text=extract_image(file)

        filetype="image"

    else:

        continue

    metadata={

        "source":file,
        "type":filetype,
        "text":text
    }

    documents.append(metadata)

In [ ]:
documents[0]

{'source': 'POLICY_WORDINGS_b6b468ad34 (2).pdf',
 'type': 'pdf',
 'text': 'दि ओरिएण्टfल इंश्योिेंस कम्पनी दलदिटेड\nTHE ORIENTAL INSURANCE COMPANY LIMITED\nपंजीकृ त कार्ाालर् :- ओरिएण्टसल हाऊस, पो.बॉ. -न 7ं 037, ए-25/27 आसफ अली िोड, नई दिल्ली r\nRegd. Office: Oriental House,A-25/27,AsafAli Road, New Delhi-110002\nCIN No.U66010DL1947GOI007158\nPOLICY FOR KISSAN AGRICULTURAL PUMPSET INSURANCE\nOperative clause\nWhereas the insured by a proposal and declaration, which shall be the basis of this contract and is\ndeemed to be incorporated herein has applied for the insurance hereunder contained and has paid\npremium as consideration for such insurance to The Oriental Insurance Company Ltd. (hereinafter\ncalled the "company"). Now this policy witnesses that subject to terms and conditions, provisions and\nexclusions contained herein or agricultural pump set described in the schedule herein (including\nstarters and switches), whilst at the premises as mentioned in the schedule herein be lost, 

###TASK 2 — Multimodal Query Support

Install dependencies and Import libraries

In [ ]:
!pip install -q pytesseract
!pip install -q pillow
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q faiss-cpu

In [ ]:
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [ ]:
import pytesseract
import numpy as np
import faiss

from PIL import Image
from google.colab import files

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

from sentence_transformers import SentenceTransformer

Upload image

In [ ]:
uploaded = files.upload()

Saving Screenshot 2026-05-26 110743.png to Screenshot 2026-05-26 110743.png


OCR extraction function

In [ ]:
def extract_text(image_path):

    image=Image.open(image_path)

    text=pytesseract.image_to_string(image)

    return text

Load BLIP model

In [ ]:
processor=BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model=BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Generate image caption

In [ ]:
image_path="Screenshot 2026-05-26 110743.png"

image=Image.open(
    image_path
).convert("RGB")

inputs=processor(
    image,
    return_tensors="pt"
)

output=model.generate(**inputs)

caption=processor.decode(
    output[0],
    skip_special_tokens=True
)

print("Caption:")
print(caption)

Caption:
what we expect to be a 1 - 1 - 1 - 1 - 1 - 1 - 1 -


Combine OCR + BLIP output

In [ ]:
ocr_text = extract_text(image_path)
final_text=ocr_text+"\n"+caption

print(final_text)

What We Expect from You — Before 4:15 PM Today

Review TCS's key platforms: BaANCS™, ADD™, ignio™, WisdomNext™, Quartz™, Cognix™
Identify 2-3 domains where your academic projects or skills are most relevant
Practice at least 5-7 DSA problems (medium difficulty) in your preferred language

Prepare a concise, confident response to: 'Which TCS domain do you see yourself contributing
to, and why?"

Dress formally and carry printed copies of your updated resume

what we expect to be a 1 - 1 - 1 - 1 - 1 - 1 - 1 -


Create embeddings

In [ ]:
embedding_model=SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embedding=embedding_model.encode(
    [final_text]
)

print(
    embedding.shape
)

(1, 384)


Store into FAISS

In [ ]:
dimension=embedding.shape[1]

index=faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(
        embedding
    )
)

print(
    "Stored successfully"
)

Stored successfully


Test image retrieval

In [ ]:
query="claim amount"

query_embedding=embedding_model.encode(
    [query]
)

distance,indices=index.search(
    np.array(query_embedding),
    1
)

print(final_text)

What We Expect from You — Before 4:15 PM Today

Review TCS's key platforms: BaANCS™, ADD™, ignio™, WisdomNext™, Quartz™, Cognix™
Identify 2-3 domains where your academic projects or skills are most relevant
Practice at least 5-7 DSA problems (medium difficulty) in your preferred language

Prepare a concise, confident response to: 'Which TCS domain do you see yourself contributing
to, and why?"

Dress formally and carry printed copies of your updated resume

what we expect to be a 1 - 1 - 1 - 1 - 1 - 1 - 1 -


###Task 3 — Multilingual Query Support (PDF only)

Install dependencies and Import libraries

In [ ]:
!pip install -q pdfplumber
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q langchain

In [ ]:
import pdfplumber
import numpy as np
import faiss

from google.colab import files

from sentence_transformers import SentenceTransformer

!pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

Upload PDF

In [ ]:
uploaded=files.upload()

Saving POLICY_WORDINGS_b6b468ad34.pdf to POLICY_WORDINGS_b6b468ad34 (3).pdf


PDF text extraction

In [ ]:
def extract_pdf(file):

    text=""

    with pdfplumber.open(file) as pdf:

        for page in pdf.pages:

            page_text=page.extract_text()

            if page_text:

                text += page_text + "\n"

    return text

Read PDF

In [ ]:
pdf_text=extract_pdf(
    "POLICY_WORDINGS_b6b468ad34.pdf"
)

print(pdf_text[:1000])

दि ओरिएण्टfल इंश्योिेंस कम्पनी दलदिटेड
THE ORIENTAL INSURANCE COMPANY LIMITED
पंजीकृ त कार्ाालर् :- ओरिएण्टसल हाऊस, पो.बॉ. -न 7ं 037, ए-25/27 आसफ अली िोड, नई दिल्ली r
Regd. Office: Oriental House,A-25/27,AsafAli Road, New Delhi-110002
CIN No.U66010DL1947GOI007158
POLICY FOR KISSAN AGRICULTURAL PUMPSET INSURANCE
Operative clause
Whereas the insured by a proposal and declaration, which shall be the basis of this contract and is
deemed to be incorporated herein has applied for the insurance hereunder contained and has paid
premium as consideration for such insurance to The Oriental Insurance Company Ltd. (hereinafter
called the "company"). Now this policy witnesses that subject to terms and conditions, provisions and
exclusions contained herein or agricultural pump set described in the schedule herein (including
starters and switches), whilst at the premises as mentioned in the schedule herein be lost, damaged or
destroyed by :
1. Fire and/or lightning.
2. Theft/burglary due to violent fo

Split PDF into chunks

In [ ]:
splitter=RecursiveCharacterTextSplitter(

    chunk_size=500,
    chunk_overlap=100
)

chunks=splitter.split_text(
    pdf_text
)

print(
    "Total Chunks:",
    len(chunks)
)

Total Chunks: 55


Load multilingual embedding model

In [ ]:
model=SentenceTransformer(
    "sentence-transformers/LaBSE"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generate embeddings

In [ ]:
embeddings=model.encode(
    chunks
)

embeddings=np.array(
    embeddings
)

print(
    embeddings.shape
)

(55, 768)


Create FAISS vector database

In [ ]:
dimension=embeddings.shape[1]

index=faiss.IndexFlatL2(
    dimension
)

index.add(
    embeddings
)

print(
    "FAISS index created"
)

FAISS index created


Create chunk mapping

In [ ]:
chunk_mapping=dict(

    zip(
        range(len(chunks)),
        chunks
    )
)

Multilingual retrieval function

In [ ]:
def search(query):

    query_embedding=model.encode(
        [query]
    )

    distance,indices=index.search(
        np.array(query_embedding),
        3
    )

    print("Query:")
    print(query)

    print("\nRetrieved Results:\n")

    # Filter out invalid indices (e.g., -1) before iteration
    valid_indices = [idx for idx in indices[0] if idx >= 0]

    for i in valid_indices:
        print(chunk_mapping[i])

        print("="*50)

Test with English

In [ ]:
search(
    "What is insurance policy?"
)

Query:
What is insurance policy?

Retrieved Results:

without being repaired to the satisfaction of the company.
3. If the proposal or declaration of the insured is not true in any material respect or if any claim made
by fraudulent or substantially exaggerated or if any false declaration or statement be made in
support thereof, this policy shall be void and the company shall not be liable to make payment
hereunder.
In the event of company disclaiming liability in respect of any claim, if an action or suit be not
iii) If any property insured which shall be removed from the premises in which it is herein
stated to be safe so far as is expressly provided for in this policy or this endorsement.
iv) If any property the interest of the Insured in which shall pass from the insured otherwise
than by will or operation of law.
Unless in every case the consent of the Company to the continuance of the insurance thereon is
obtaining and signified by a Memorandum made on the Policy by or on behalf 

Test with Hindi

In [ ]:
search(
    "बीमा दावा प्रक्रिया क्या है?"
)

Query:
बीमा दावा प्रक्रिया क्या है?

Retrieved Results:

without being repaired to the satisfaction of the company.
3. If the proposal or declaration of the insured is not true in any material respect or if any claim made
by fraudulent or substantially exaggerated or if any false declaration or statement be made in
support thereof, this policy shall be void and the company shall not be liable to make payment
hereunder.
In the event of company disclaiming liability in respect of any claim, if an action or suit be not
In any action, suit or other proceeding where the Company alleges that by reasons of the
provisions of the Exception above and loss, destruction, damage or liability is not covered by this
insurance the burden of proving that such loss, destruction, damage or liability is covered shall be
upon the Insured.
CONDITIONS
1. a. The company shall at all reasonable times have the right to inspect and examine any property
insured hereunder.
damage (if any) to the Premises. The Insu

Test with French

In [ ]:
search(
    "Qu'est-ce qu'une police d'assurance?"
)

Query:
Qu'est-ce qu'une police d'assurance?

Retrieved Results:

iii) If any property insured which shall be removed from the premises in which it is herein
stated to be safe so far as is expressly provided for in this policy or this endorsement.
iv) If any property the interest of the Insured in which shall pass from the insured otherwise
than by will or operation of law.
Unless in every case the consent of the Company to the continuance of the insurance thereon is
obtaining and signified by a Memorandum made on the Policy by or on behalf of the
Company.
without being repaired to the satisfaction of the company.
3. If the proposal or declaration of the insured is not true in any material respect or if any claim made
by fraudulent or substantially exaggerated or if any false declaration or statement be made in
support thereof, this policy shall be void and the company shall not be liable to make payment
hereunder.
In the event of company disclaiming liability in respect of any claim, i

Test with German

In [ ]:
search(
    "Was ist eine Versicherungspolice?"
)

Query:
Was ist eine Versicherungspolice?

Retrieved Results:

without being repaired to the satisfaction of the company.
3. If the proposal or declaration of the insured is not true in any material respect or if any claim made
by fraudulent or substantially exaggerated or if any false declaration or statement be made in
support thereof, this policy shall be void and the company shall not be liable to make payment
hereunder.
In the event of company disclaiming liability in respect of any claim, if an action or suit be not
iii) If any property insured which shall be removed from the premises in which it is herein
stated to be safe so far as is expressly provided for in this policy or this endorsement.
iv) If any property the interest of the Insured in which shall pass from the insured otherwise
than by will or operation of law.
Unless in every case the consent of the Company to the continuance of the insurance thereon is
obtaining and signified by a Memorandum made on the Policy by or on

Task 4 — AI Evaluation Agent

Install dependencies and Import libraries

In [ ]:
!pip install -q sentence-transformers
!pip install -q scikit-learn

In [ ]:
import time
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

Load evaluation model

In [ ]:
evaluation_model=SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample inputs

In [ ]:
question="What is insurance premium?"

retrieved_context="""
Insurance premium is the amount paid
periodically by the customer to maintain
coverage.
"""

generated_answer="""
Insurance premium is the amount paid
periodically by a customer for insurance coverage.
"""

Measure latency

In [ ]:
start=time.time()

# simulate LLM response generation
answer=generated_answer

end=time.time()

latency=end-start

print(
    "Latency:",
    latency
)

Latency: 9.012222290039062e-05


Faithfulness score

In [ ]:
def faithfulness(context,answer):

    embeddings=evaluation_model.encode(
        [context,answer]
    )

    score=cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(float(score),3)

Test

In [ ]:
print(
    "Faithfulness:",
    faithfulness(
        retrieved_context,
        generated_answer
    )
)

Faithfulness: 0.981


Relevancy score

In [ ]:
def relevancy(question,answer):

    embeddings=evaluation_model.encode(
        [question,answer]
    )

    score=cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(float(score),3)

Test

In [ ]:
print(
    "Relevancy:",
    relevancy(
        question,
        generated_answer
    )
)

Relevancy: 0.881


Groundedness score

In [ ]:
def groundedness(context,answer):

    embeddings=evaluation_model.encode(
        [context,answer]
    )

    score=cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return round(float(score),3)

Test


In [ ]:
print(
    groundedness(
        retrieved_context,
        generated_answer
    )
)

0.981


Hallucination detection

In [ ]:
def detect_hallucination(
        context,
        answer
):

    score=groundedness(
        context,
        answer
    )

    if score<0.50:

        return "Hallucination Detected"

    else:

        return "Grounded Response"

Test

In [ ]:
print(
    detect_hallucination(
        retrieved_context,
        generated_answer
    )
)

Grounded Response


Complete evaluation agent

In [ ]:
def evaluation_agent(
        question,
        context,
        answer,
        latency
):

    result={}

    result["Faithfulness"]=faithfulness(
        context,
        answer
    )

    result["Relevancy"]=relevancy(
        question,
        answer
    )

    result["Groundedness"]=groundedness(
        context,
        answer
    )

    result["Latency"]=round(
        latency,
        4
    )

    result["Hallucination"]=detect_hallucination(
        context,
        answer
    )

    return result

Run final evaluation

In [ ]:
scores=evaluation_agent(

    question,
    retrieved_context,
    generated_answer,
    latency
)

print(scores)

{'Faithfulness': 0.981, 'Relevancy': 0.881, 'Groundedness': 0.981, 'Latency': 0.0001, 'Hallucination': 'Grounded Response'}


###Conclusion

This project successfully extends a traditional Retrieval-Augmented Generation (RAG) system into an Enterprise AI Knowledge Platform by integrating advanced capabilities such as multi-source data ingestion, multimodal understanding, multilingual retrieval, and automated response evaluation. The system was designed to process and retrieve information from various enterprise data sources including PDF files, DOCX documents, websites, and images while maintaining metadata-aware retrieval for better contextual understanding.

The implementation of multimodal support enabled the system to understand image-based content through OCR and image captioning techniques, allowing it to process scanned documents and visual information effectively. Multilingual support was achieved using multilingual embedding models, enabling users to query the system in different languages and retrieve relevant information from English documents through cross-language semantic understanding.

An Evaluation Agent was also incorporated to assess the quality and reliability of generated responses using metrics such as faithfulness, relevancy, groundedness, and latency. This helped in reducing hallucinations and improving trust in AI-generated outputs.

Overall, the project demonstrates how modern enterprise AI systems can evolve beyond basic retrieval pipelines into intelligent, scalable, and production-oriented knowledge platforms capable of serving real-world enterprise requirements.